# Runpod에서 실행 대기 실험 시작

먼저 README.md의 GPU·Network Volume 설정을 따르세요. 이 노트북은 `/workspace/runpod_start`에서 엽니다.

|JOB|대상|실행 범위|
|---|---|---|
|`dh-ngcf`|Dunnhumby NGCF|SEEDS에 지정한 시드 × 8모형. seed49는 기존 원본 재사용|
|`hm-ngcf`|H&M NGCF|seed49 × 8모형|
|`hm-lightgcn`|H&M LightGCN|seed51 × 8모형|

진행 중인 학교 GCN·GraphSAGE 작업과 분리합니다. 동일 작업이 다른 곳에서 시작됐으면 이곳에서 새 학습을 시작하지 마세요. 과거 Colab 체크포인트를 이 출력에 복사하는 기능은 제공하지 않습니다.

**모든 학습은 고정300epoch, 마지막1주 신규상품 test, M1~M5_D 8모형입니다.**

## 1. 처음 한 번 설치 — 학습 없음
Python 3.10~3.12의 공식 Runpod PyTorch/JupyterLab 템플릿을 사용하세요. 독립 환경에 PyTorch 2.8.0 CUDA12.8과 필요한 패키지를 설치합니다. 이미 이 패키지로 실행한 Pod 재접속 시 설치 셀은 건너뛰세요.

In [ ]:
from pathlib import Path
import subprocess, sys, json, datetime
PACKAGE = Path('/workspace/runpod_start')
assert (PACKAGE/'launch.py').is_file(), 'runpod_start.zip을 /workspace에 압축 해제하세요.'
subprocess.run(['bash', str(PACKAGE/'setup.sh')], check=True)


## 2. 작업 선택과 입력 설치
기본값은 Dunnhumby NGCF seed42의 8모형입니다. 남은9시드를 모두 실행하려면 SEEDS를 `[42,43,44,45,46,47,48,50,51]`로 지정합니다. H&M은 정해진 시드만 사용하며 SEEDS가 자동으로 고정됩니다.

H&M은 학교 Drive의 `논문/data/raw/hm/transactions_train.parquet`를 직접 다운로드하여 `/workspace/runpod_start`에 업로드하세요. Colab을 쓰지 않아도 됩니다. 또는 동봉한 CPU 내보내기 노트북으로 만든 `hm_raw_verified.zip`을 업로드해도 됩니다.

In [ ]:
from pathlib import Path
import subprocess, json, datetime
PACKAGE = Path('/workspace/runpod_start')
PYTHON = '/workspace/clv/venv/bin/python'
JOB = 'dh-ngcf'  # 'hm-ngcf' 또는 'hm-lightgcn'
SEEDS = [42] if JOB == 'dh-ngcf' else ([49] if JOB == 'hm-ngcf' else [51])
for name in ['dh_inputs.zip', 'hm_support.zip', 'hm_raw_verified.zip']:
    bundle = PACKAGE/name
    if bundle.is_file():
        subprocess.run([PYTHON, str(PACKAGE/'launch.py'), 'unpack', '--bundle', str(bundle)], check=True)
raw = PACKAGE/'transactions_train.parquet'
if raw.is_file():
    subprocess.run([PYTHON, str(PACKAGE/'launch.py'), 'hm-raw', '--file', str(raw)], check=True)
BASE = [PYTHON, '-u', str(PACKAGE/'launch.py')]
OPTIONS = ['--job', JOB, '--seeds', *map(str, SEEDS)]
print('선택:', JOB, SEEDS)


## 3. 원본·GPU·코드 사전점검 — 학습 없음
파일 해시, 고정 코드, CUDA 희소행렬 역전파를 확인합니다. 실제 전체 전처리·메모리 최고 사용량·학습 속도는 다음 실행에서 확인됩니다.

In [ ]:
subprocess.run(BASE + ['check'] + OPTIONS, check=True)


## 4. 학습 시작 또는 이 Runpod 실행 재개
**같은 모델·시드가 다른 코랩/서버에서 실행 중이거나 완료됐으면 실행하지 마세요.**

아래 셀은 별도 프로세스로 시작해 브라우저를 닫아도 Pod가 켜져 있는 동안 계속 실행합니다. 완료 모형은 재학습·재평가하지 않고, 중단된 학습은 같은 Runpod 출력의 마지막 저장 epoch에서 이어갑니다. 저장 폴더·원본·GPU·패키지·실행도구가 달라지면 자동 재개하지 않습니다.

In [ ]:
log_dir = Path('/workspace/clv/logs')
log_dir.mkdir(parents=True, exist_ok=True)
log_path = log_dir/(JOB + '_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f') + '.log')
with log_path.open('a') as log:
    process = subprocess.Popen(BASE + ['run'] + OPTIONS + ['--confirmed-unstarted'],
        stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print('프로세스:', process.pid, '| 로그:', log_path)
print('시작 요청입니다. 아래 로그에서 입력 검증과 첫 epoch 완료를 확인하세요.')


## 5. 진행 로그 확인
오류가 나면 출력 폴더를 삭제하지 마세요. 최종평가 시작 후 결과 저장 전에 중단된 경우에는 자동 재평가가 차단됩니다. 원본 결과 복구를 먼저 확인해야 합니다.

In [ ]:
if 'log_path' not in globals():
    candidates = sorted(Path('/workspace/clv/logs').glob(JOB + '_*.log'))
    assert candidates, '로그가 없습니다.'
    log_path = candidates[-1]
subprocess.run(['tail', '-n', '45', str(log_path)], check=True)
if 'process' in globals():
    print('실행 중' if process.poll() is None else f'종료 코드: {process.returncode}')


## 6. 결과 받기와 종료
모형 하나가 끝날 때마다 `/workspace/clv/exports`에 ZIP이 갱신됩니다. JupyterLab 파일 목록에서 우클릭 → Download 하세요. `_partial`은 전체 계획 미완료이며 최종10시드 평균이 아닙니다. ZIP에는 전체 지표·진단·설정·학습이력·환경 기록이 들어갑니다.

재개용 가중치·optimizer는 ZIP에 넣지 않고 `/workspace/clv/results`에 최신·직전 파일로 보존합니다. Network Volume을 유지하세요. 별도 Mac 백업은 자동이 아닙니다. **작업이 끝나도 요금은 계속 발생하므로 결과를 받은 뒤 Runpod 콘솔에서 Stop 하세요.** Network Volume 저장요금은 별도입니다.

In [ ]:
for p in sorted(Path('/workspace/clv/exports').glob('*.zip')):
    print(p, f'{p.stat().st_size/1024**2:.2f} MiB')
